# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and analyzing the FAIR² extracellular vesicle proteomics dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset URL: {croissant_url}")

## 2. Data Overview

Review available record sets and fields. Record set, field, and column details are all referenced by their `@id` fields for reproducibility and future-proof usage.

In [ ]:
# List all record sets and their fields with @id references

record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    if hasattr(rs, 'description') and rs.description:
        print(f"  Description: {rs.description}")
    fields = rs.fields
    print("  Fields and their @id:")
    for field in fields:
        print(f"    - {field['@id']} ({field.get('name', '')})")
    print()

Below is a sample of the first few rows from the first available record set, referenced by its `@id`.

In [ ]:
# Show some records with @id referencing

if record_sets:
    selected_record_set = record_sets[0].id
    print(f"Iterating records from record_set: {selected_record_set}")
    for i, rec in enumerate(dataset.records(record_set=selected_record_set)):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction

Load data from record sets into DataFrames for analysis, using each record set's `@id`. The columns in the DataFrame correspond to field `@id` values.

In [ ]:
# Extract all data into per-recordset DataFrames using @id

dataframes = {}
for rs in record_sets:
    rs_id = rs.id
    print(f"Loading record set {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns (@id):", df.columns.tolist())
    if not df.empty:
        display(df.head(2))

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like filtering, normalizing, or grouping by `@id` fields.

In [ ]:
# EDA Example: Filtering and normalizing a numeric field

# Auto-select recordset and numeric field if available
import numpy as np

main_rs_id = record_sets[0].id if record_sets else None
df = dataframes[main_rs_id]

# Find candidate numeric fields (float/int columns)
numeric_fields = []
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_fields.append(col)

if numeric_fields:
    numeric_field = numeric_fields[0]  # Pick the first one for demo
    print(f"Using numeric field for EDA: {numeric_field}")
    threshold = df[numeric_field].mean()  # Use mean as sample threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered {len(filtered_df)}/{len(df)} records with {numeric_field} > {threshold:.2f}")

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Select a group field (categorical) if available
    group_field = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and df[col].nunique() < 10 and col != numeric_field:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped records by {group_field}, mean {numeric_field} values:")
        display(grouped_df.head())
else:
    print("No numeric field detected for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. All visualizations use columns referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if numeric_fields:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(9,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

- Loaded FAIR² mass spectrometry dataset via Croissant schema and `mlcroissant`.
- Referenced all entities (record sets, fields, columns) by their `@id` per best practice.
- Performed initial exploratory analysis, with numeric field filtering, normalization, and visualizations.
- Use specific field names and `@id`s as required for downstream or domain-specific analyses.